# Task A Run 31 full-data final -- Gemma-4-12B + TAPT

This is the full-data submission companion to `31_gemma_tapt.ipynb`. It first performs
Gemma TAPT on the labelled Task A comments plus the external Kannada corpus, then
fine-tunes four Gemma classifiers on **all 7,193 labelled Task A rows**. It averages
their predictions for the 806 released test comments and writes:

`RECOMMENDED_a31_gemma_tapt.zip` containing a bare `predictions.csv`.

There is no holdout score in this notebook: all labelled rows are used for training.
The TAPT stage never reads the test comments. Use GPU T4 x2, Internet on, and run
all cells. The TAPT stage is run before classifier jobs so its GPU memory is isolated.
If TAPT cannot complete after its memory-safe retry, the notebook stops rather than
silently producing a non-TAPT submission under this experiment's name.

In [ ]:
import json, os, pathlib, subprocess, sys, time
from fnmatch import fnmatch

T_START = time.time()
HARD_STOP = T_START + 11 * 3600
REPO = 'https://github.com/robinpnalex/Hastika-ICON2026.git'
BRANCH = 'task-b'
WORK = '/tmp/hastika'

def git(*args):
    return subprocess.run(['git', '-C', WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

remote = subprocess.run(['git', 'ls-remote', '--exit-code', REPO, f'refs/heads/{BRANCH}'],
                         check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
if os.path.isdir(WORK + '/.git'):
    git('fetch', '-q', '--depth', '1', 'origin', BRANCH)
    git('reset', '-q', '--hard', 'FETCH_HEAD')
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, '--depth', '1', REPO, WORK], check=True)
head = git('rev-parse', 'HEAD')
assert head == remote, f'clone is at {head[:8]} but {BRANCH} is at {remote[:8]}'
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))
print('repo at', git('log', '-1', '--oneline'))
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf \"transformers>=4.45,<6\"', shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), 'select GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('gpus:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
OUT = pathlib.Path('/kaggle/working/a31_gemma_tapt_full_outputs')
RUNS, LOGS, SUB = OUT / 'runs', OUT / 'logs', OUT / 'submissions'
for p in (RUNS, LOGS, SUB):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = '/tmp/hf'
os.environ['HF_HUB_CACHE'] = HF_CACHE
from hastika.common.gpu_queue import run_queue


In [ ]:
LLM_PY = '/tmp/llmenv/bin/python'
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', '/tmp/llmenv'], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q \"transformers>=5.12,<6\" \"peft>=0.17\" \"bitsandbytes>=0.46\" \"accelerate>=1.0\" emoji ftfy sentencepiece protobuf', shell=True, check=True)
subprocess.run([LLM_PY, '-c', "import transformers, peft, bitsandbytes; print('LLM env:', transformers.__version__, peft.__version__)"], check=True)

MODEL = 'google/gemma-4-12B'
TASK = 'a'
CLASSES = ['Non-Hate', 'Hate']
ALL = pathlib.Path(WORK) / 'data/derived/task_a_combined_holdout/all.csv'
EXTERNAL = pathlib.Path(WORK) / 'data/external/offenseval_kn.csv'
TEST_CSV = pathlib.Path('/tmp/a31_test_inputs.csv')
test_source = pathlib.Path(WORK) / 'data/raw/hastika_binary_test.csv'
pd.read_csv(test_source)[['id', 'Comment']].to_csv(TEST_CSV, index=False)
df = pd.read_csv(ALL)
test = pd.read_csv(TEST_CSV)
assert len(df) == 7193, len(df)
assert len(test) == 806, len(test)
print(f'full Task A rows: {len(df)}; test rows: {len(test)}')

def download():
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get('HF_TOKEN') or None,
                      allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt', 'tokenizer*', '*.jinja'])

download()


In [ ]:
TAPT_DIR = RUNS / 'tapt_all'

def tapt_job(name, bs, grad_accum, logit_chunk):
    texts = ' '.join(str(p) for p in (ALL, EXTERNAL))
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_tapt --model {MODEL} '
           f'--texts {texts} --out {TAPT_DIR} --cache {HF_CACHE} '
           f'--bs {bs} --grad-accum {grad_accum} --logit-chunk {logit_chunk}')
    return {'name': name, 'log': str(LOGS / f'{name}.log'), 'cmd': cmd}

def classifier_job(seed, name=None, extra=''):
    name = name or f'full_s{seed}'
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_classifier --task {TASK} --model {MODEL} '
           f'--train {ALL} --predict {TEST_CSV} --out {RUNS / name} --seed {seed} '
           f'--epochs 3 --prompt short --lr 1e-4 --class-weight balanced '
           f'--init-adapter {TAPT_DIR} --cache {HF_CACHE} {extra}')
    return {'name': name, 'log': str(LOGS / f'{name}.log'), 'cmd': cmd}

def run_tapt():
    first = tapt_job('tapt_all', bs=2, grad_accum=8, logit_chunk=32)
    codes = run_queue([first], N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
    code = codes.get('tapt_all', 1)
    if code == 0 and (TAPT_DIR / 'adapter_config.json').exists():
        return codes
    print('TAPT first attempt failed; retrying with bs=1 and logit chunks=16')
    retry = tapt_job('tapt_all_retry', bs=1, grad_accum=16, logit_chunk=16)
    retry_codes = run_queue([retry], N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
    retry_code = retry_codes.get('tapt_all_retry', 1)
    assert retry_code == 0 and (TAPT_DIR / 'adapter_config.json').exists(), 'TAPT did not finish; inspect the TAPT logs before producing a submission'
    return {**codes, **retry_codes}


In [ ]:
tapt_codes = run_tapt()
print('TAPT exit codes:', tapt_codes)
assert (TAPT_DIR / 'adapter_config.json').exists()

SEEDS = [42, 43, 44, 45]
registry = {j['name']: j for j in [classifier_job(s) for s in SEEDS]}
retried = set()

def classifier_done(name, code):
    if not code or name in retried:
        return []
    job = registry[name]
    log = pathlib.Path(job['log']).read_text(errors='replace') if pathlib.Path(job['log']).exists() else ''
    import re
    seed_match = re.search(r'--seed (\d+)', job['cmd'])
    if code == 4 and seed_match:
        seed = int(seed_match.group(1)) + 1000
        retry = classifier_job(seed, name=name)
        print(f'{name} collapsed; retrying with seed {seed}')
    elif 'OutOfMemoryError' in log:
        retry = classifier_job(int(seed_match.group(1)), name=name, extra='--bs 4 --grad-accum 4 --eval-bs 8')
        print(f'{name} ran out of memory; retrying with smaller classifier batch')
    else:
        return []
    retried.add(name)
    registry[name] = retry
    return [retry]

classifier_codes = run_queue(list(registry.values()), N_GPU, raise_on_fail=False,
                             on_done=classifier_done, stop_at=HARD_STOP)
print('classifier exit codes:', classifier_codes)
print(f'elapsed: {(time.time() - T_START) / 3600:.1f} h')


In [ ]:
def load_probs(name):
    run = RUNS / name
    p = run / f'{TEST_CSV.stem}_probs.npy'
    ids = run / f'{TEST_CSV.stem}_ids.csv'
    if not p.exists() or not ids.exists():
        return None
    got_ids = pd.read_csv(ids)['id'].tolist()
    if got_ids != test['id'].tolist():
        return None
    probs = np.load(p)
    share = np.bincount(probs.argmax(1), minlength=len(CLASSES)) / len(probs)
    if share.max() > 0.9:
        print(f'skipping collapsed {name}: {share.max():.0%} in one class')
        return None
    return probs

members = {s: load_probs(f'full_s{s}') for s in SEEDS}
members = {s: p for s, p in members.items() if p is not None}
assert members, 'no healthy classifier finished; inspect the classifier logs'
probs = np.mean(list(members.values()), axis=0)
pred = pd.DataFrame({'id': test['id'], 'label': [CLASSES[i] for i in probs.argmax(1)]})
pred_path = SUB / 'predictions_a31_gemma_tapt.csv'
pred.to_csv(pred_path, index=False)
np.save(SUB / 'a31_gemma_tapt_test_probs.npy', probs)
import zipfile
zip_path = SUB / 'RECOMMENDED_a31_gemma_tapt.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, 'predictions.csv')
json.dump({'commit': head, 'tapt': True, 'epochs': 3, 'prompt': 'short',
           'seeds': list(members), 'tapt_exit_codes': tapt_codes,
           'classifier_exit_codes': classifier_codes, 'zip': str(zip_path)},
          open(OUT / 'result.json', 'w'), indent=2)
print('healthy classifier seeds:', list(members))
print('predicted shares:', pred['label'].value_counts(normalize=True).to_dict())
print('READY TO UPLOAD:', zip_path)
